# Signum Execution Policy Verification

Visual + numerical audit of the four execution policies for the CACT momentum signal.

| Policy | `execution=` | Lag | Fill price | Returns earned while long |
|--------|-------------|-----|------------|---------------------------|
| 0 | `0` | 0 | close[T] after signal at close[T] (precise execution) | cc_ret every bar |
| 1 | `1` | 1 | close[T+1] after signal at close[T] | cc_ret every bar |
| NO | `"NO"` | 1 | open[T+1] after signal at close[T] | Entry bar: oc_ret; holding bars: cc_ret |
| 2 | `2` | 2 | close[T+2] after signal at close[T] | cc_ret every bar |

In [1]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import sfera_db
from signum import Chart, Dashboard
from signum.engine.chart import Chart as _Chart


_HERE       = pathlib.Path('.').resolve()
_BTEST_ROOT = _HERE.parents[1]
_WS_ROOT    = _BTEST_ROOT.parent

for _p in [
    str(_WS_ROOT),
    str(_WS_ROOT / 'sfera-db'),
    str(_WS_ROOT / 'signum'),
    str(_BTEST_ROOT / 'src'),
]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

def _q(sql):
    return (sfera_db.query(sql)
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .set_index('date'))

cactr = _q("SELECT trade_date AS date, open_price AS open, close_price AS close "
           "FROM bbgidx.index_total_return WHERE ticker = 'CACT' ORDER BY trade_date")

df = cactr[['open', 'close']].copy()
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
df = df.dropna().copy()

print(f'Data: {df.index[0].date()} -> {df.index[-1].date()} ({len(df)} days)')

Data: 2000-01-04 -> 2026-04-17 (6724 days)


In [2]:
MOM_WINDOW  = 139
THRESHOLD   = 0.005
TEST_START  = "2022-04-01"
TEST_END    = "2022-04-30"
CARRY_IN    = True

def _to_lc(series, col_name="value"):
    return series.rename(col_name).reset_index().rename(columns={"date": "time"})

mom_f    = np.log(df["close"] / df["close"].shift(MOM_WINDOW))
cc_ret   = np.exp(df["log_ret"]) - 1
oc_ret   = df["close"] / df["open"] - 1
gate_raw = (mom_f >= THRESHOLD).astype(float)

sl   = slice(TEST_START, TEST_END)
df_t = df.loc[sl]
mom_t  = mom_f.loc[sl]
gate_t = gate_raw.loc[sl]
cc_t   = cc_ret.loc[sl]
oc_t   = oc_ret.loc[sl]

# Warm-start: use full history so pre-period signal shifts in correctly.
_g, _cc, _oc = (gate_raw, cc_ret, oc_ret) if CARRY_IN else (gate_t, cc_t, oc_t)

# apply_execution with correct API: int or "NO"
r_same = _Chart.apply_execution(_g, _cc, execution=0,    carry_in=CARRY_IN)
r_moc  = _Chart.apply_execution(_g, _cc, execution=1,    carry_in=CARRY_IN)
r_moo  = _Chart.apply_execution(_g, _cc, execution="NO", open_returns=_oc, carry_in=CARRY_IN)
r_lag2 = _Chart.apply_execution(_g, _cc, execution=2,    carry_in=CARRY_IN)

# Period-only slices
r_same_t = r_same.loc[sl]
r_moc_t  = r_moc.loc[sl]
r_moo_t  = r_moo.loc[sl]
r_lag2_t = r_lag2.loc[sl]

def _nav(r):
    return 100 * (1 + r).cumprod()
nav_same = _nav(r_same_t)
nav_moc  = _nav(r_moc_t)
nav_moo  = _nav(r_moo_t)
nav_lag2 = _nav(r_lag2_t)
nav_bh   = _nav(cc_t)

print(f"Test period returns  (carry_in={CARRY_IN}):")
for lbl, r in [("exec=0 same_bar", r_same_t), ("exec=1 MOC     ", r_moc_t),
               ("exec=NO MOO    ", r_moo_t),   ("exec=2 lag-2   ", r_lag2_t)]:
    tot = (1+r).prod()-1
    print(f"  {lbl}  {tot:+.2%}")
print(f"  B&H              {(1+cc_t).prod()-1:+.2%}")

Test period returns  (carry_in=True):
  exec=0 same_bar  -1.91%
  exec=1 MOC       -4.67%
  exec=NO MOO      -0.61%
  exec=2 lag-2     -2.44%
  B&H              -1.27%


In [3]:
# Manual cross-validation: shift = execution + 1
# exec=0: signal at T, fill at close[T], earn cc_ret[T+1] → shift(1)
# exec=1: signal at T, fill at close[T+1], earn cc_ret[T+2] → shift(2)
# exec="NO": signal at T, fill at open[T+1], entry=oc_ret, holding=cc_ret → shift(1)
# exec=2: signal at T, fill at close[T+2], earn cc_ret[T+3] → shift(3)
r_same_man  = (gate_raw.shift(1) * cc_ret).loc[sl]
r_moc_man   = (gate_raw.shift(2) * cc_ret).loc[sl]
# "NO" mode: oc_ret on entry bars, cc_ret on holding bars
_s_moo = gate_raw.shift(1)
_e_moo = (_s_moo > 0) & (_s_moo.shift(1).fillna(0) <= 0)
_b_moo = cc_ret.copy(); _b_moo[_e_moo] = oc_ret[_e_moo]
r_moo_man   = (_s_moo * _b_moo).loc[sl]
r_lag2_man  = (gate_raw.shift(3) * cc_ret).loc[sl]

print('Cross-validation: apply_execution() vs manual pandas recompute')
print(f'{"Policy":<28} {"Manual total":>13} {"Signum total":>13} {"Max|diff|":>12}  Match')
print('-'*74)
for lbl, r_man, r_sg in [
    ('exec=0 same_bar',        r_same_man,  r_same_t),
    ('exec=1 MOC',             r_moc_man,   r_moc_t),
    ('exec="NO" MOO (blended)',r_moo_man,   r_moo_t),
    ('exec=2 lag-2',           r_lag2_man,  r_lag2_t),
]:
    tot_m = (1 + r_man).prod() - 1
    tot_s = (1 + r_sg).prod() - 1
    mx    = (r_man - r_sg).abs().max()
    ok    = 'EXACT' if mx < 1e-12 else f'MISMATCH  max={mx:.3e}'
    print(f'{lbl:<28} {tot_m:>+12.4%} {tot_s:>+12.4%} {mx:>12.3e}   {ok}')

Cross-validation: apply_execution() vs manual pandas recompute
Policy                        Manual total  Signum total    Max|diff|  Match
--------------------------------------------------------------------------
exec=0 same_bar                  -1.9140%     -1.9140%    0.000e+00   EXACT
exec=1 MOC                       -4.6701%     -4.6701%    0.000e+00   EXACT
exec="NO" MOO (blended)          -0.6063%     -0.6063%    0.000e+00   EXACT
exec=2 lag-2                     -2.4449%     -2.4449%    0.000e+00   EXACT


In [4]:
def show_policy(nav_series, ret_series, signal_series, exec_mode, label, color):
    """Chart + stats for one execution policy."""
    nav = (1 + ret_series).cumprod()
    n_y = len(ret_series) / 252
    cagr = nav.iloc[-1] ** (1 / n_y) - 1
    vol = ret_series.std(ddof=1) * np.sqrt(252)
    sr = cagr / vol if vol > 0 else 0
    mdd = (nav / nav.cummax() - 1).min()
    days = int((ret_series != 0).sum())
    tot = nav.iloc[-1] - 1

    p_eq = (Chart(theme='dark', height=220)
        .line(_to_lc(nav_series), name=label, color=color)
        .line(_to_lc(nav_bh), name='B&H', color='#666666')
        .stats_legend({
            'Return': f'{tot:+.2%}',
            'CAGR': f'{cagr:+.2%}',
            'Vol': f'{vol:.2%}',
            'Sharpe': f'{sr:.2f}',
            'Max DD': f'{mdd:.2%}',
            'In Mkt': f'{days}/{len(ret_series)}',
        }))
    p_sig = (Chart(theme='dark', height=120)
        .line(_to_lc(signal_series.astype(float)), name='signal (raw gate)', color=color))
    Dashboard(
        panes=[p_eq, p_sig],
        titles=[f'NAV $100 | {label}  [{TEST_START} - {TEST_END}]', 'Signal (0/1 raw gate)'],
        theme='dark',
        execution=exec_mode,
    ).show()

bh_nav = (1 + cc_t).cumprod()
bh_cagr = bh_nav.iloc[-1] ** (1 / (len(cc_t)/252)) - 1
bh_vol = cc_t.std(ddof=1) * np.sqrt(252)
print(f'B&H reference [{TEST_START} - {TEST_END}]')
print(f'  Return={bh_nav.iloc[-1]-1:+.2%}  CAGR={bh_cagr:+.2%}  Vol={bh_vol:.2%}')

B&H reference [2022-04-01 - 2022-04-30]
  Return=-1.27%  CAGR=-15.62%  Vol=17.95%


In [5]:
show_policy(nav_same, r_same_t, gate_t, 0, 'exec=0 fill@close[T]', '#ff6b6b')

In [6]:
show_policy(nav_moc, r_moc_t, gate_t, 1, 'exec=1 MOC fill@close[T+1]', '#2a9d8f')

In [7]:
show_policy(nav_moo, r_moo_t, gate_t, "NO", 'exec="NO" MOO fill@open[T+1]', '#f4a261')

In [8]:
show_policy(nav_lag2, r_lag2_t, gate_t, 2, 'exec=2 fill@close[T+2]', '#a29bfe')

In [9]:
import importlib, signum.engine.dashboard, signum.engine.chart, signum
importlib.reload(signum.engine.chart)
importlib.reload(signum.engine.dashboard)
importlib.reload(signum)
from signum import Chart, Dashboard
from signum.engine.chart import Chart as _Chart

sig_df  = (mom_f.loc[TEST_START:TEST_END] if CARRY_IN
           else mom_t).to_frame('value')
prices_src = df

ret_map = {0: r_same_t, 1: r_moc_t, "NO": r_moo_t, 2: r_lag2_t}

print(f'Slider dashboards  theta={THRESHOLD}  [{TEST_START} - {TEST_END}]  carry_in={CARRY_IN}')

theta_min = round(float(mom_t.min()) - 0.0025, 4)
theta_max = round(float(mom_t.max()) + 0.0025, 4)
step      = 0.0025

def slider_chart(exec_mode, color, label):
    r = ret_map[exec_mode]
    nav = (1 + r).cumprod()
    tot = nav.iloc[-1] - 1
    print(f'  {label}  NAV={100*(1+tot):.2f}  return={tot:+.2%}')

    p_eq  = Chart(theme='dark', height=220)
    p_sig = (Chart(theme='dark', height=120)
        .baseline(_to_lc(mom_t), base_value=THRESHOLD,
                  name=f'Momentum({MOM_WINDOW})',
                  topLineColor=color,
                  topFillColor1='rgba(100,100,255,0.08)', topFillColor2='transparent',
                  bottomLineColor='rgba(140,140,140,0.5)',
                  bottomFillColor1='transparent', bottomFillColor2='transparent'))
    return (Dashboard(
                panes=[p_eq, p_sig],
                titles=[f'SLIDER | {label}  [{TEST_START}-{TEST_END}]  carry_in={CARRY_IN}',
                        f'Momentum({MOM_WINDOW}) raw vs theta'],
                theme='dark',
                execution=exec_mode)
            .threshold_control(
                df=sig_df,
                signal_mode='>=',
                threshold=THRESHOLD,
                min_val=theta_min,
                max_val=theta_max,
                step=step,
                price_pane=1,
                equity_pane=0,
                prices=prices_src,
                strategy_color=color,
                bh_color='#666666',
                equity_clip_start=TEST_START,
                carry_in=CARRY_IN,
            ))

slider_chart(0,    '#ff6b6b', 'exec=0 fill@close[T]').show()
slider_chart(1,    '#2a9d8f', 'exec=1 MOC fill@close[T+1]').show()
slider_chart("NO", '#f4a261', 'exec="NO" MOO fill@open[T+1]').show()
slider_chart(2,    '#a29bfe', 'exec=2 fill@close[T+2]').show()

Slider dashboards  theta=0.005  [2022-04-01 - 2022-04-30]  carry_in=True
  exec=0 fill@close[T]  NAV=98.09  return=-1.91%


  exec=1 MOC fill@close[T+1]  NAV=95.33  return=-4.67%


  exec="NO" MOO fill@open[T+1]  NAV=99.39  return=-0.61%


  exec=2 fill@close[T+2]  NAV=97.56  return=-2.44%


In [10]:
# Cross-check: recompute NAVs from scratch — must match Cell 3 exactly
_cases = [
    (0,    "exec=0 same_bar",   nav_same, {}),
    (1,    "exec=1 MOC",        nav_moc,  {}),
    ("NO", 'exec="NO" MOO',     nav_moo,  {"open_returns": oc_ret}),
    (2,    "exec=2 lag-2",      nav_lag2, {}),
]

print(f"Cross-check: apply_execution carry_in={CARRY_IN} vs reference NAVs")
print(f'{"Policy":<20} {"recomputed":>12} {"reference":>12} {"diff":>12}  Match')
print("-"*64)
all_ok = True
for exec_mode, label, nav_ref, kw in _cases:
    _r    = _Chart.apply_execution(gate_raw, cc_ret, execution=exec_mode,
                                   carry_in=CARRY_IN, **kw)
    _nc   = 100.0 * (1 + _r.loc[TEST_START:TEST_END]).cumprod()
    diff  = abs(_nc.iloc[-1] - nav_ref.iloc[-1])
    ok    = "EXACT" if diff < 1e-8 else f"MISMATCH diff={diff:.3e}"
    if diff >= 1e-8: all_ok = False
    print(f"{label:<20} {_nc.iloc[-1]:>12.4f} {nav_ref.iloc[-1]:>12.4f} {diff:>12.3e}   {ok}")
print()
print("ALL CHECKS PASSED" if all_ok else "WARNING: mismatch found.")


Cross-check: apply_execution carry_in=True vs reference NAVs
Policy                 recomputed    reference         diff  Match
----------------------------------------------------------------
exec=0 same_bar           98.0860      98.0860    0.000e+00   EXACT
exec=1 MOC                95.3299      95.3299    0.000e+00   EXACT
exec="NO" MOO             99.3937      99.3937    0.000e+00   EXACT
exec=2 lag-2              97.5551      97.5551    0.000e+00   EXACT

ALL CHECKS PASSED


In [11]:
INITIAL_NAV = 100.0
pd.set_option("display.float_format", '{:.6f}'.format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)

def audit(exec_mode, label, use_oc=False):
    lag = {0: 1, 1: 2, "NO": 1, 2: 3}[exec_mode]
    r = _Chart.apply_execution(gate_raw, cc_ret, execution=exec_mode,
                               carry_in=CARRY_IN,
                               **({'open_returns': oc_ret} if use_oc else {}))
    pos = gate_raw.shift(lag).loc[sl]
    fill_price = df_t["open"] if use_oc else df_t["close"]
    entry = (pos == 1) & (pos.shift(1).fillna(0) == 0)
    cost_src = fill_price.copy()
    if CARRY_IN and pos.iloc[0] == 1:
        cost_src.iloc[0] = df_t["open"].iloc[0]
    avg_cost = cost_src.where(entry).ffill().where(pos == 1)
    out = pd.DataFrame({
        "open": df_t["open"], "close": df_t["close"],
        "mom": mom_t, "gate": gate_t.astype(int), "pos": pos.astype(int),
        "avg_cost": avg_cost,
        "cc_ret": cc_t,
        "strat_ret": r.loc[sl],
        "nav": INITIAL_NAV * (1 + r.loc[sl]).cumprod(),
    })
    out.index = out.index.strftime('%Y-%m-%d')
    out.index.name = 'date'
    return out

print(f'exec=0 same_bar (shift=1, cc_ret)')
display(audit(0, 'same_bar'))
print(f'\nexec=1 MOC (shift=2, cc_ret)')
display(audit(1, 'MOC'))
print(f'\nexec="NO" MOO (shift=1, entry=oc, hold=cc)')
display(audit("NO", 'MOO', use_oc=True))
print(f'\nexec=2 lag-2 (shift=3, cc_ret)')
display(audit(2, 'lag-2'))

exec=0 same_bar (shift=1, cc_ret)


,open,close,mom,gate,pos,avg_cost,cc_ret,strat_ret,nav
date,,,,,,,,,
2022-04-01,19035.100000,19067.440000,0.040683,1,1,19035.100000,0.003669,0.003669,100.366939
2022-04-04,19139.580000,19201.670000,0.031721,1,1,19035.100000,0.007040,0.007040,101.073497
2022-04-05,19201.600000,18956.770000,0.006107,1,1,19035.100000,-0.012754,-0.012754,99.784395
2022-04-06,18926.300000,18538.330000,-0.025957,0,1,19035.100000,-0.022073,-0.022073,97.581816
2022-04-07,18629.600000,18432.370000,-0.022166,0,0,NaN,-0.005716,-0.000000,97.581816
2022-04-08,18656.360000,18679.220000,-0.010737,0,0,NaN,0.013392,0.000000,97.581816
2022-04-11,18708.650000,18700.880000,0.011220,1,0,NaN,0.001160,0.000000,97.581816
2022-04-12,18411.160000,18648.390000,0.000098,0,1,18648.390000,-0.002807,-0.002807,97.307922
2022-04-13,18596.320000,18661.880000,0.007058,1,0,NaN,0.000723,0.000000,97.307922



exec=1 MOC (shift=2, cc_ret)


,open,close,mom,gate,pos,avg_cost,cc_ret,strat_ret,nav
date,,,,,,,,,
2022-04-01,19035.100000,19067.440000,0.040683,1,1,19035.100000,0.003669,0.003669,100.366939
2022-04-04,19139.580000,19201.670000,0.031721,1,1,19035.100000,0.007040,0.007040,101.073497
2022-04-05,19201.600000,18956.770000,0.006107,1,1,19035.100000,-0.012754,-0.012754,99.784395
2022-04-06,18926.300000,18538.330000,-0.025957,0,1,19035.100000,-0.022073,-0.022073,97.581816
2022-04-07,18629.600000,18432.370000,-0.022166,0,1,19035.100000,-0.005716,-0.005716,97.024066
2022-04-08,18656.360000,18679.220000,-0.010737,0,0,NaN,0.013392,0.000000,97.024066
2022-04-11,18708.650000,18700.880000,0.011220,1,0,NaN,0.001160,0.000000,97.024066
2022-04-12,18411.160000,18648.390000,0.000098,0,0,NaN,-0.002807,-0.000000,97.024066
2022-04-13,18596.320000,18661.880000,0.007058,1,1,18661.880000,0.000723,0.000723,97.094251



exec="NO" MOO (shift=1, entry=oc, hold=cc)


,open,close,mom,gate,pos,avg_cost,cc_ret,strat_ret,nav
date,,,,,,,,,
2022-04-01,19035.100000,19067.440000,0.040683,1,1,19035.100000,0.003669,0.003669,100.366939
2022-04-04,19139.580000,19201.670000,0.031721,1,1,19035.100000,0.007040,0.007040,101.073497
2022-04-05,19201.600000,18956.770000,0.006107,1,1,19035.100000,-0.012754,-0.012754,99.784395
2022-04-06,18926.300000,18538.330000,-0.025957,0,1,19035.100000,-0.022073,-0.022073,97.581816
2022-04-07,18629.600000,18432.370000,-0.022166,0,0,NaN,-0.005716,-0.000000,97.581816
2022-04-08,18656.360000,18679.220000,-0.010737,0,0,NaN,0.013392,0.000000,97.581816
2022-04-11,18708.650000,18700.880000,0.011220,1,0,NaN,0.001160,0.000000,97.581816
2022-04-12,18411.160000,18648.390000,0.000098,0,1,18411.160000,-0.002807,0.012885,98.839170
2022-04-13,18596.320000,18661.880000,0.007058,1,0,NaN,0.000723,0.000000,98.839170



exec=2 lag-2 (shift=3, cc_ret)


,open,close,mom,gate,pos,avg_cost,cc_ret,strat_ret,nav
date,,,,,,,,,
2022-04-01,19035.100000,19067.440000,0.040683,1,1,19035.100000,0.003669,0.003669,100.366939
2022-04-04,19139.580000,19201.670000,0.031721,1,1,19035.100000,0.007040,0.007040,101.073497
2022-04-05,19201.600000,18956.770000,0.006107,1,1,19035.100000,-0.012754,-0.012754,99.784395
2022-04-06,18926.300000,18538.330000,-0.025957,0,1,19035.100000,-0.022073,-0.022073,97.581816
2022-04-07,18629.600000,18432.370000,-0.022166,0,1,19035.100000,-0.005716,-0.005716,97.024066
2022-04-08,18656.360000,18679.220000,-0.010737,0,1,19035.100000,0.013392,0.013392,98.323431
2022-04-11,18708.650000,18700.880000,0.011220,1,0,NaN,0.001160,0.000000,98.323431
2022-04-12,18411.160000,18648.390000,0.000098,0,0,NaN,-0.002807,-0.000000,98.323431
2022-04-13,18596.320000,18661.880000,0.007058,1,0,NaN,0.000723,0.000000,98.323431
